# Heart Disease Classifier - Exploratory Data Analysis

This notebook investigates the Heart Failure Prediction Dataset before preprocessing and model training. The raw dataset is examined, not modified. Findings will inform the preprocessing pipeline, project documentation, and a potential dataset/EDA web section.

## 1. EDA Objectives

The workflow checks dataset structure, types, missing values, duplicates, category consistency, suspicious values, distributions, feature-target relationships, correlations, and IQR outliers. Flagged values are not automatically errors and no cleaning occurs here.

## 2. Imports and Configuration

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

pio.templates.default = 'plotly_white'
TARGET = 'HeartDisease'
NUMERICAL_FEATURES = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
CONTINUOUS_FEATURES = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
CATEGORICAL_FEATURES = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
TARGET_LABELS = {0: 'Normal', 1: 'Heart disease'}
COLOR_MAP = {'Normal': '#4C78A8', 'Heart disease': '#E45756'}

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'ml' / 'data' / 'raw' / 'heart.csv').is_file():
            return candidate
    raise FileNotFoundError('Could not locate ml/data/raw/heart.csv from the current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / 'ml' / 'data' / 'raw' / 'heart.csv'
df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset from: {DATA_PATH.relative_to(PROJECT_ROOT)}')

Loaded dataset from: ml\data\raw\heart.csv


## 3. Load Dataset

The dataset path is relative to this notebook. df is intentionally not modified.

## 4. Dataset Overview

In [2]:
print(f'Shape: {df.shape}')
print(f'Observations: {len(df)} | Input features: {df.shape[1] - 1}')
print('Columns:', df.columns.tolist())
display(df.head())
display(df.tail())
df.info()
display(df.describe(include='all').T)

Shape: (918, 12)
Observations: 918 | Input features: 11
Columns: ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1
917,38,M,NAP,138,175,0,Normal,173,N,0.0,Up,0


<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 86.2 KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,918.0,NaN,NaN,NaN,53.510893,9.432617,28.0,47.0,54.0,60.0,77.0
Sex,918,2,M,725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ChestPainType,918,4,ASY,496,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RestingBP,918.0,NaN,NaN,NaN,132.396514,18.514154,0.0,120.0,130.0,140.0,200.0
Cholesterol,918.0,NaN,NaN,NaN,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
FastingBS,918.0,NaN,NaN,NaN,0.233115,0.423046,0.0,0.0,0.0,0.0,1.0
RestingECG,918,3,Normal,552,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MaxHR,918.0,NaN,NaN,NaN,136.809368,25.460334,60.0,120.0,138.0,156.0,202.0
ExerciseAngina,918,2,N,547,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Oldpeak,918.0,NaN,NaN,NaN,0.887364,1.06657,-2.6,0.0,0.6,1.5,6.2


## 5. Data Types

In [3]:
type_table = pd.DataFrame({'Feature': df.columns, 'Data type': df.dtypes.astype(str).values})
type_table['Role'] = np.where(type_table['Feature'].eq(TARGET), 'Target', 'Feature')
type_table['Feature type'] = type_table['Feature'].map(lambda x: 'Categorical (binary)' if x == TARGET else ('Categorical' if x in CATEGORICAL_FEATURES else 'Numerical'))
display(type_table)

,Feature,Data type,Role,Feature type
0,Age,int64,Feature,Numerical
1,Sex,str,Feature,Categorical
2,ChestPainType,str,Feature,Categorical
3,RestingBP,int64,Feature,Numerical
4,Cholesterol,int64,Feature,Numerical
5,FastingBS,int64,Feature,Categorical
6,RestingECG,str,Feature,Categorical
7,MaxHR,int64,Feature,Numerical
8,ExerciseAngina,str,Feature,Categorical
9,Oldpeak,float64,Feature,Numerical


## 6. Missing Values

NaN values and numeric zeros are distinct checks. Zero values are not replaced with NaN in this notebook.

In [4]:
missing = df.isnull().sum().rename('Missing values').to_frame()
display(missing)
print('Total missing/NaN cells:', int(missing.values.sum()))
print('No NaN values were detected.' if missing.values.sum() == 0 else 'NaN values require investigation.')

,Missing values
Age,0
Sex,0
ChestPainType,0
RestingBP,0
Cholesterol,0
FastingBS,0
RestingECG,0
MaxHR,0
ExerciseAngina,0
Oldpeak,0


Total missing/NaN cells: 0
No NaN values were detected.


## 7. Duplicate Analysis

In [5]:
duplicates = int(df.duplicated().sum())
print(f'Duplicate rows: {duplicates}')
print('Duplicate rows are not currently present.' if duplicates == 0 else 'Duplicate rows require investigation.')

Duplicate rows: 0
Duplicate rows are not currently present.


## 8. Unique Values and Category Validation

Observed categories are compared with the expected schema without modification.

In [6]:
EXPECTED = {'Sex': {'M','F'}, 'ChestPainType': {'TA','ATA','NAP','ASY'}, 'FastingBS': {0,1}, 'RestingECG': {'Normal','ST','LVH'}, 'ExerciseAngina': {'Y','N'}, 'ST_Slope': {'Up','Flat','Down'}}
audit_rows = []
for feature in CATEGORICAL_FEATURES:
    observed = set(df[feature].unique())
    audit_rows.append({'Feature': feature, 'Unique values': df[feature].nunique(), 'Observed': ', '.join(map(str, sorted(observed))), 'Unexpected': ', '.join(map(str, sorted(observed - EXPECTED[feature]))) or 'None', 'Expected absent': ', '.join(map(str, sorted(EXPECTED[feature] - observed))) or 'None'})
    print(f'\n{feature}')
    display(df[feature].value_counts(dropna=False).rename('Count').to_frame())
display(pd.DataFrame(audit_rows))


Sex


,Count
Sex,
M,725
F,193



ChestPainType


,Count
ChestPainType,
ASY,496
NAP,203
ATA,173
TA,46



FastingBS


...


ExerciseAngina


,Count
ExerciseAngina,
N,547
Y,371



ST_Slope


,Count
ST_Slope,
Flat,460
Up,395
Down,63


,Feature,Unique values,Observed,Unexpected,Expected absent
0,Sex,2,"F, M",None,None
1,ChestPainType,4,"ASY, ATA, NAP, TA",None,None
2,FastingBS,2,"0, 1",None,None
3,RestingECG,3,"LVH, Normal, ST",None,None
4,ExerciseAngina,2,"N, Y",None,None
5,ST_Slope,3,"Down, Flat, Up",None,None


## 9. Numerical Summary Statistics

In [7]:
display(df[NUMERICAL_FEATURES].describe().T.style.format(precision=2))
print('Context: Age mean is near 53; RestingBP is concentrated around 120-140 for many records; MaxHR commonly falls around 120-160; cholesterol is strongly affected by zero values.')

,count,mean,std,min,25%,50%,75%,max
Age,918.00,53.51,9.43,28.00,47.00,54.00,60.00,77.00
RestingBP,918.00,132.40,18.51,0.00,120.00,130.00,140.00,200.00
Cholesterol,918.00,198.80,109.38,0.00,173.25,223.00,267.00,603.00
FastingBS,918.00,0.23,0.42,0.00,0.00,0.00,0.00,1.00
MaxHR,918.00,136.81,25.46,60.00,120.00,138.00,156.00,202.00
Oldpeak,918.00,0.89,1.07,-2.60,0.00,0.60,1.50,6.20


Context: Age mean is near 53; RestingBP is concentrated around 120-140 for many records; MaxHR commonly falls around 120-160; cholesterol is strongly affected by zero values.


## 10. Suspicious Zero Values

A zero can be valid for one feature but suspicious or unrecorded information for another. Cholesterol zeros need a future decision; Oldpeak zeros are not automatically errors.

In [8]:
ZERO_FEATURES = ['RestingBP', 'Cholesterol', 'Oldpeak']
zero_summary = pd.DataFrame({'Feature': ZERO_FEATURES, 'Zero Count': [(df[x] == 0).sum() for x in ZERO_FEATURES]})
zero_summary['Percentage of Dataset'] = (zero_summary['Zero Count'] / len(df) * 100).round(2)
display(zero_summary)
print('RestingBP = 0 observation:')
display(df.loc[df['RestingBP'].eq(0), ['Age','Sex','ChestPainType','RestingBP','Cholesterol',TARGET]])

,Feature,Zero Count,Percentage of Dataset
0,RestingBP,1,0.11
1,Cholesterol,172,18.74
2,Oldpeak,368,40.09


RestingBP = 0 observation:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,HeartDisease
449,55,M,NAP,0,0,1


## 11. Negative Value Analysis

In [9]:
negative = pd.DataFrame({'Feature': CONTINUOUS_FEATURES, 'Negative Count': [(df[x] < 0).sum() for x in CONTINUOUS_FEATURES]})
display(negative)
print('Negative Oldpeak values require contextual interpretation before preprocessing:')
display(df.loc[df['Oldpeak'].lt(0)])

,Feature,Negative Count
0,Age,0
1,RestingBP,0
2,Cholesterol,0
3,MaxHR,0
4,Oldpeak,13


Negative Oldpeak values require contextual interpretation before preprocessing:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
321,63,M,ASY,100,0,1,Normal,109,N,-0.9,Flat,1
324,46,M,ASY,100,0,1,ST,133,N,-2.6,Flat,1
325,42,M,ASY,105,0,1,Normal,128,Y,-1.5,Down,1
326,45,M,NAP,110,0,0,Normal,138,N,-0.1,Up,0
331,56,M,ASY,115,0,1,ST,82,N,-1.0,Up,1
332,38,M,NAP,100,0,0,Normal,179,N,-1.1,Up,0
334,51,M,ASY,130,0,1,Normal,170,N,-0.7,Up,1
335,62,M,TA,120,0,1,LVH,134,N,-0.8,Flat,1
352,56,M,ASY,120,0,0,ST,100,Y,-1.0,Down,1
407,62,M,ASY,115,0,1,Normal,72,Y,-0.5,Flat,1


## 12. Target Distribution

In [10]:
target_distribution = df[TARGET].value_counts().sort_index().rename_axis(TARGET).reset_index(name='Count')
target_distribution['Label'] = target_distribution[TARGET].map(TARGET_LABELS)
target_distribution['Percentage'] = (target_distribution['Count'] / len(df) * 100).round(2)
display(target_distribution[['Label','Count','Percentage']])
fig = px.bar(target_distribution, x='Label', y='Count', color='Label', color_discrete_map=COLOR_MAP, text=target_distribution.apply(lambda row: f"{row['Count']} ({row['Percentage']:.2f}%)", axis=1), title='HeartDisease Target Distribution')
fig.update_layout(xaxis_title='Target class', yaxis_title='Observations', showlegend=False)
fig.show()

,Label,Count,Percentage
0,Normal,410,44.66
1,Heart disease,508,55.34


## 13. Numerical Feature Distributions

Histograms compare target groups. Cholesterol zeros and negative Oldpeak values are retained so they remain visible.

In [11]:
plot_df = df.assign(HeartDiseaseLabel=df[TARGET].map(TARGET_LABELS))
for feature in CONTINUOUS_FEATURES:
    fig = px.histogram(plot_df, x=feature, color='HeartDiseaseLabel', barmode='overlay', opacity=0.65, nbins=30, color_discrete_map=COLOR_MAP, title=f'{feature} Distribution by HeartDisease', labels={'HeartDiseaseLabel':'HeartDisease'})
    fig.update_layout(xaxis_title=feature, yaxis_title='Observations', legend_title_text='Target')
    fig.show()

## 14. Categorical Feature Distributions

In [12]:
for feature in ['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']:
    counts = df[feature].value_counts().rename_axis(feature).reset_index(name='Count')
    fig = px.bar(counts, x=feature, y='Count', text='Count', title=f'{feature} Distribution')
    fig.update_layout(xaxis_title=feature, yaxis_title='Observations', showlegend=False)
    fig.show()

## 15. Feature vs Target Analysis

These comparisons are observational and do not establish causation or statistical significance.

In [13]:
for feature in CONTINUOUS_FEATURES:
    fig = px.box(plot_df, x='HeartDiseaseLabel', y=feature, color='HeartDiseaseLabel', color_discrete_map=COLOR_MAP, title=f'{feature} by HeartDisease', labels={'HeartDiseaseLabel':'HeartDisease'})
    fig.update_layout(xaxis_title='Target class', yaxis_title=feature, showlegend=False)
    fig.show()

for feature in ['ChestPainType','ExerciseAngina','ST_Slope','Sex','RestingECG']:
    rel = pd.crosstab(df[feature], df[TARGET], normalize='index').mul(100).rename(columns=TARGET_LABELS).reset_index().melt(id_vars=feature, var_name='HeartDisease', value_name='Percentage')
    fig = px.bar(rel, x=feature, y='Percentage', color='HeartDisease', barmode='stack', color_discrete_map=COLOR_MAP, text=rel['Percentage'].round(1), title=f'{feature} vs HeartDisease (within-category percentage)')
    fig.update_layout(xaxis_title=feature, yaxis_title='Percentage of category', legend_title_text='Target')
    fig.show()

## 16. Correlation Analysis

Correlation measures association, not causation.

In [14]:
correlations = df[NUMERICAL_FEATURES + [TARGET]].corr()
display(correlations.round(2))
fig = px.imshow(correlations, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Numerical Feature Correlation Heatmap')
fig.update_layout(coloraxis_colorbar_title='Correlation')
fig.show()
print('Initial audit associations with HeartDisease: Oldpeak about +0.40, Age about +0.28, and MaxHR about -0.40.')

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
Age,1.00,0.25,-0.10,0.20,-0.38,0.26,0.28
RestingBP,0.25,1.00,0.10,0.07,-0.11,0.16,0.11
Cholesterol,-0.10,0.10,1.00,-0.26,0.24,0.05,-0.23
FastingBS,0.20,0.07,-0.26,1.00,-0.13,0.05,0.27
MaxHR,-0.38,-0.11,0.24,-0.13,1.00,-0.16,-0.40
Oldpeak,0.26,0.16,0.05,0.05,-0.16,1.00,0.40
HeartDisease,0.28,0.11,-0.23,0.27,-0.40,0.40,1.00


Initial audit associations with HeartDisease: Oldpeak about +0.40, Age about +0.28, and MaxHR about -0.40.


## 17. Outlier Analysis

IQR flags are a statistical screening mechanism, not proof of invalid data. FastingBS is binary, so its IQR result needs special interpretation.

In [15]:
def iqr_summary(frame, features):
    rows = []
    for feature in features:
        q1, q3 = frame[feature].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        rows.append({'Feature': feature, 'Q1': q1, 'Q3': q3, 'IQR': iqr, 'Lower Bound': lower, 'Upper Bound': upper, 'Outlier Count': int(((frame[feature] < lower) | (frame[feature] > upper)).sum())})
    return pd.DataFrame(rows)

display(iqr_summary(df, CONTINUOUS_FEATURES).style.format(precision=2))
display(iqr_summary(df, ['FastingBS']).style.format(precision=2))
for feature in CONTINUOUS_FEATURES:
    fig = px.box(df, y=feature, points='outliers', title=f'IQR Screening: {feature}')
    fig.update_layout(yaxis_title=feature, showlegend=False)
    fig.show()

ARTIFACT_DIR = PROJECT_ROOT / 'ml' / 'artifacts' / 'eda'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
target_distribution.to_json(ARTIFACT_DIR / 'target_distribution.json', orient='records', indent=2)
correlations.to_json(ARTIFACT_DIR / 'correlation_matrix.json', indent=2)
iqr_summary(df, CONTINUOUS_FEATURES + ['FastingBS']).to_json(ARTIFACT_DIR / 'iqr_outlier_summary.json', orient='records', indent=2)
category_counts = {feature: df[feature].value_counts().to_dict() for feature in CATEGORICAL_FEATURES}
(ARTIFACT_DIR / 'categorical_value_counts.json').write_text(json.dumps(category_counts, indent=2), encoding='utf-8')
eda_summary = {'shape': list(df.shape), 'missing_nan_cells': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()), 'zero_counts': {feature: int((df[feature] == 0).sum()) for feature in ZERO_FEATURES}, 'negative_counts': {feature: int((df[feature] < 0).sum()) for feature in CONTINUOUS_FEATURES}}
(ARTIFACT_DIR / 'eda_summary.json').write_text(json.dumps(eda_summary, indent=2), encoding='utf-8')
print(f'Wrote reusable EDA artifacts to: {ARTIFACT_DIR.relative_to(PROJECT_ROOT)}')

,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Age,47.00,60.00,13.00,27.50,79.50,0
1,RestingBP,120.00,140.00,20.00,90.00,170.00,28
2,Cholesterol,173.25,267.00,93.75,32.62,407.62,183
3,MaxHR,120.00,156.00,36.00,66.00,210.00,2
4,Oldpeak,0.00,1.50,1.50,-2.25,3.75,16


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,FastingBS,0.00,0.00,0.00,0.00,0.00,214


Wrote reusable EDA artifacts to: ml\artifacts\eda


## 18. Important Data Quality Findings

- No NaN values were found, but numeric zeros can still be suspicious.
- Cholesterol equals zero in 172 observations and requires a future preprocessing decision; no treatment occurs here.
- One RestingBP observation equals zero and is retained for investigation.
- Thirteen Oldpeak observations are negative and require contextual analysis rather than automatic removal.
- The target contains 410 Normal and 508 Heart disease observations, a moderate rather than severe imbalance.
- IQR flags are not automatically bad data.

## 19. EDA Conclusions

### Confirmed observations
- 918 rows, 12 columns, no NaN values, and no duplicate rows.
- The target distribution is 410 Normal and 508 Heart disease.

### Issues requiring preprocessing decisions
- Treatment of Cholesterol = 0 and RestingBP = 0.
- Categorical encoding, scaling where appropriate, and evaluation strategy.

### Findings requiring further investigation
- Negative Oldpeak values, IQR flags, and the contextual meaning of suspicious numeric values.

## 20. Preprocessing Recommendations

Future decisions only: decide how to treat Cholesterol = 0 and RestingBP = 0; investigate negative Oldpeak values; choose categorical encoding and scaling; define a leakage-safe train/validation/test split; and consider target imbalance during evaluation.

## 21. Next Steps

1. Finalize EDA findings.
2. Decide data-cleaning strategy.
3. Build preprocessing pipeline.
4. Split data correctly.
5. Train baseline models.
6. Evaluate multiple models.
7. Select a final model.
8. Save model and preprocessing artifacts.
9. Build backend inference API.
10. Build frontend and landing page.
11. Expose selected EDA findings and visualizations on the website.